모델 입력 tensor를 생성. 

feature 제외, imputation, scaling, categorical encoding 값은 train split에서만 fit

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

RANDOM_STATE = 42
MISSING_THRESHOLD = 0.95

SRC_DIR = Path.cwd().resolve()
PROJECT_DIR = SRC_DIR.parent
DATA_SPLIT_DIR = PROJECT_DIR / "processed" / "data_split"
CLEAN_DATA_DIR = PROJECT_DIR / "models" / "clean_data"
CATALOG_PATH = SRC_DIR / "extraction_variable_catalog.csv"

In [2]:
required_files = {
    "binned": DATA_SPLIT_DIR / "events_12h_binned_with_split.csv",
    "seq_train": DATA_SPLIT_DIR / "lstm_sequence_index_train.csv",
    "seq_test": DATA_SPLIT_DIR / "lstm_sequence_index_test.csv",
}
binned = pd.read_csv(required_files["binned"])
seq_train = pd.read_csv(required_files["seq_train"])
seq_test = pd.read_csv(required_files["seq_test"])
catalog = pd.read_csv(CATALOG_PATH)
seq_all = pd.concat([seq_train, seq_test], ignore_index=True)

for df_name, df in [("binned", binned), ("seq_train", seq_train), ("seq_test", seq_test)]:
    print(df_name, df.shape)



binned (7653, 172)
seq_train (4934, 23)
seq_test (1944, 23)


## Feature set 정의

In [3]:
leakage_or_metadata_cols = {
    "subject_id", "hadm_id", "stay_id", "bin", "bin_start", "bin_end",
    "split", "intime", "outtime", "delirium", "ever_delirium", "los_hours",
    "admission_type", "specialty",
}

catalog_feature_types = catalog.drop_duplicates("feature_name").set_index("feature_name")["type"].to_dict()
catalog_feature_types["prev_delirium"] = "binary"
catalog_feature_types["race"] = "categorical"

candidate_cols = [col for col in binned.columns if col not in leakage_or_metadata_cols]
catalog_type_by_col = {col: catalog_feature_types.get(col, "numeric") for col in candidate_cols}
categorical_cols = [col for col in candidate_cols if catalog_type_by_col[col] == "categorical"]
binary_cols = [col for col in candidate_cols if catalog_type_by_col[col] == "binary"]
numeric_cols = [col for col in candidate_cols if col not in categorical_cols + binary_cols]

print("categorical_cols", categorical_cols)
print("binary_cols kept", len(binary_cols))
print("numeric_cols before missingness filter", len(numeric_cols))


categorical_cols ['race']
binary_cols kept 15
numeric_cols before missingness filter 143


## Train 기준 preprocessing fit

In [4]:
train_rows = binned["split"] == "train"

결측치 확인

In [6]:
numeric_missingness = binned.loc[train_rows, numeric_cols].isna().mean().sort_values(ascending=False)
dropped_high_missing = numeric_missingness[numeric_missingness > MISSING_THRESHOLD].index.tolist()
numeric_cols = [col for col in numeric_cols if col not in dropped_high_missing]

print("numeric_cols kept", len(numeric_cols))
print("numeric_cols dropped for high missingness", len(dropped_high_missing))
display(numeric_missingness)

numeric_cols kept 140
numeric_cols dropped for high missingness 0


bilirubin_direct     0.941793
cortisol             0.932755
glucose_std          0.866414
sao2_mean            0.819234
sao2_count           0.819234
                       ...   
heart_rate_max       0.000000
heart_rate_latest    0.000000
spo2_max             0.000000
spo2_min             0.000000
spo2_latest          0.000000
Length: 140, dtype: float64

In [ ]:
numeric_train = binned.loc[train_rows, numeric_cols]
numeric_medians = numeric_train.median(numeric_only=True).fillna(0.0)
numeric_means = numeric_train.fillna(numeric_medians).mean()
numeric_stds = numeric_train.fillna(numeric_medians).std(ddof=0).replace(0, 1.0).fillna(1.0)

binary_numeric_cols = []
binary_text_cols = []
for col in binary_cols:
    values = pd.to_numeric(binned.loc[train_rows, col], errors="coerce")
    if values.notna().any():
        binary_numeric_cols.append(col)
    else:
        binary_text_cols.append(col)

binary_text_levels = {}
for col in binary_text_cols:
    binary_text_levels[col] = sorted(binned.loc[train_rows, col].fillna("Unknown").astype(str).unique().tolist())

category_levels = {}
for col in categorical_cols:
    values = binned.loc[train_rows, col].fillna("Unknown").astype(str)
    levels = sorted(values.unique().tolist())
    if "Unknown" not in levels:
        levels.append("Unknown")
    category_levels[col] = levels


def transform_binned_features(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    if numeric_cols:
        numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
        numeric = numeric.fillna(numeric_medians)
        numeric = (numeric - numeric_means) / numeric_stds
        numeric = numeric.astype("float32")
        parts.append(numeric)

    if binary_numeric_cols:
        binary = df[binary_numeric_cols].apply(pd.to_numeric, errors="coerce")
        binary = binary.fillna(0).astype("float32")
        parts.append(binary)

    binary_text_parts = []
    for col in binary_text_cols:
        levels = binary_text_levels[col]
        values = df[col].fillna("Unknown").astype(str)
        encoded = pd.DataFrame(0.0, index=df.index, columns=[f"{col}__{level}" for level in levels])
        for level in levels:
            encoded.loc[values == level, f"{col}__{level}"] = 1.0
        binary_text_parts.append(encoded.astype("float32"))
    parts.extend(binary_text_parts)

    cat_parts = []
    for col in categorical_cols:
        levels = category_levels[col]
        known = set(levels)
        values = df[col].fillna("Unknown").astype(str)
        values = values.where(values.isin(known), "Unknown")
        encoded = pd.DataFrame(0.0, index=df.index, columns=[f"{col}__{level}" for level in levels])
        for level in levels:
            encoded.loc[values == level, f"{col}__{level}"] = 1.0
        cat_parts.append(encoded.astype("float32"))
    parts.extend(cat_parts)
    return pd.concat(parts, axis=1)


feature_frame = transform_binned_features(binned)
feature_columns = feature_frame.columns.tolist()
feature_lookup = feature_frame.copy()
feature_lookup[["stay_id", "bin"]] = binned[["stay_id", "bin"]]
feature_lookup = feature_lookup.set_index(["stay_id", "bin"]).sort_index()

print("feature_frame", feature_frame.shape)
print("first features", feature_columns[:20])

## multicollinearity 확인

## Sequence tensor 생성

In [ ]:
def parse_bin_tokens(text: str) -> list[int | None]:
    tokens = []
    for raw in str(text).split(","):
        token = raw.strip()
        if not token:
            continue
        tokens.append(None if token.upper() == "PAD" else int(token))
    return tokens


def make_tensor(sequence_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    x_rows = []
    zero_step = np.zeros(len(feature_columns), dtype=np.float32)
    for _, row in sequence_df.iterrows():
        stay_id = row["stay_id"]
        input_bins = parse_bin_tokens(row["input_bins"])
        step_rows = []
        for bin_id in input_bins:
            if bin_id is None:
                step_rows.append(zero_step.copy())
                continue
            key = (stay_id, bin_id)
            step_rows.append(feature_lookup.loc[key, feature_columns].to_numpy(dtype=np.float32))
        x_rows.append(np.stack(step_rows, axis=0))

    x = np.stack(x_rows, axis=0).astype(np.float32)
    y_binary = sequence_df["y_any_t_to_t_plus_2"].to_numpy(dtype=np.float32)
    y_steps = sequence_df[["y_t", "y_t_plus_1", "y_t_plus_2"]].to_numpy(dtype=np.float32)
    y_step_mask = sequence_df[["y_t_mask", "y_t_plus_1_mask", "y_t_plus_2_mask"]].to_numpy(dtype=np.float32)
    meta_cols = [
        "example_id", "subject_id", "hadm_id", "stay_id", "split", "n_observed_bins",
        "anchor_bin", "input_bins", "input_mask", "target_bins", "target_mask", "target_available_count",
    ]
    target_cols = [
        "y_t", "y_t_plus_1", "y_t_plus_2",
        "y_t_mask", "y_t_plus_1_mask", "y_t_plus_2_mask",
        "y_any_t_to_t_plus_2",
    ]
    meta = sequence_df[meta_cols + target_cols].copy()
    return x, y_binary, y_steps, y_step_mask, meta


X_train, y_train, y_train_steps, y_train_step_mask, meta_train = make_tensor(seq_train)
X_test, y_test, y_test_steps, y_test_step_mask, meta_test = make_tensor(seq_test)

print("X_train", X_train.shape, "y_train", y_train.shape, "positive rate", y_train.mean())
print("X_test", X_test.shape, "y_test", y_test.shape, "positive rate", y_test.mean())
print("target mask train/test", y_train_step_mask.sum(axis=0), y_test_step_mask.sum(axis=0))


## 산출물 저장

In [ ]:
np.save(DATA_SPLIT_DIR / "X_train_lstm.npy", X_train)
np.save(DATA_SPLIT_DIR / "X_test_lstm.npy", X_test)
np.save(DATA_SPLIT_DIR / "y_train_lstm.npy", y_train)
np.save(DATA_SPLIT_DIR / "y_test_lstm.npy", y_test)
np.save(DATA_SPLIT_DIR / "y_train_steps_lstm.npy", y_train_steps)
np.save(DATA_SPLIT_DIR / "y_train_step_mask_lstm.npy", y_train_step_mask)
np.save(DATA_SPLIT_DIR / "y_test_steps_lstm.npy", y_test_steps)
np.save(DATA_SPLIT_DIR / "y_test_step_mask_lstm.npy", y_test_step_mask)
meta_train.to_csv(DATA_SPLIT_DIR / "lstm_train_metadata.csv", index=False)
meta_test.to_csv(DATA_SPLIT_DIR / "lstm_test_metadata.csv", index=False)

feature_missingness_train = binned.loc[train_rows, numeric_cols].isna().mean().rename("missing_rate").reset_index().rename(columns={"index": "feature"})
feature_missingness_test = binned.loc[binned["split"] == "test", numeric_cols].isna().mean().rename("missing_rate").reset_index().rename(columns={"index": "feature"})
feature_missingness_train.to_csv(DATA_SPLIT_DIR / "feature_missingness_train.csv", index=False)
feature_missingness_test.to_csv(DATA_SPLIT_DIR / "feature_missingness_test.csv", index=False)

preprocess_params = {
    "missing_threshold": MISSING_THRESHOLD,
    "numeric_cols": numeric_cols,
    "binary_cols": binary_cols,
    "binary_numeric_cols": binary_numeric_cols,
    "binary_text_cols": binary_text_cols,
    "categorical_cols": categorical_cols,
    "dropped_high_missing": dropped_high_missing,
    "feature_columns": feature_columns,
    "category_levels": category_levels,
    "binary_text_levels": binary_text_levels,
    "target": "masked_y_t_to_t_plus_2",
    "excluded_columns": sorted(leakage_or_metadata_cols),
}
with open(CLEAN_DATA_DIR / "lstm_feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2, ensure_ascii=False)
with open(CLEAN_DATA_DIR / "lstm_preprocess_params.json", "w") as f:
    json.dump(preprocess_params, f, indent=2, ensure_ascii=False)

joblib.dump(
    {
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "category_levels": category_levels,
        "binary_text_levels": binary_text_levels,
        "numeric_cols": numeric_cols,
        "binary_cols": binary_cols,
        "binary_numeric_cols": binary_numeric_cols,
        "binary_text_cols": binary_text_cols,
        "categorical_cols": categorical_cols,
        "feature_columns": feature_columns,
    },
    CLEAN_DATA_DIR / "lstm_preprocessor.joblib",
)

summary = pd.DataFrame(
    [
        {"split": "train", "n_sequences": len(y_train), "n_positive": int(y_train.sum()), "positive_rate": float(y_train.mean())},
        {"split": "test", "n_sequences": len(y_test), "n_positive": int(y_test.sum()), "positive_rate": float(y_test.mean())},
    ]
)
summary.to_csv(DATA_SPLIT_DIR / "lstm_preprocessing_summary.csv", index=False)
display(summary)
print("Saved preprocessing outputs")